# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [18]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama

In [3]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [5]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

API key looks good so far


In [6]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [15]:
# system prompt for this agent
system_prompt = "You're an assistant, an expert in Software Engineeing, that takes a technical questions and replies with an explanation."


In [19]:
# Get gpt-4o-mini to answer, with streaming

def stream_technical_answer(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_technical_answer(question)

The provided code is using a generator expression with the `yield from` statement. Let’s break it down step by step to understand what it does and why:

1. **Context**:
   - It seems to be part of a generator function in Python. Generators are a type of iterable that generate values on the fly and can be used to handle large amounts of data efficiently.

2. **Set comprehension**:
   - The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension. This creates a set containing unique author names from a list or collection of `books`. 
   - Here's how it works:
     - `for book in books`: Iterates through each `book` in the `books` iterable.
     - `book.get("author")`: Retrieves the value associated with the key `"author"` in the `book` dictionary. If the key does not exist, it returns `None` by default.
     - `if book.get("author")`: This part is a conditional statement that checks if the author exists (i.e., it is not `None` or an empty string). The author will only be included in the set if this condition is met.

3. **Yield from**:
   - The `yield from` statement is used to yield all the values from an iterable (which, in this case, is the set created previously). It allows the generator to yield each value produced by the set comprehension one at a time.
   - This is particularly useful for delegating the iteration to another iterator and effectively returning its results, while still being part of the generator's output.

### Summary
The entire line effectively creates a generator that yields all unique authors from a collection of `books`, ignoring any entries where the author is `None` or doesn't exist. This is useful when you want to gather a distinct list of authors without duplicates, returning them one at a time without loading them all into memory at once. The use of `yield from` makes it efficient and suitable for large datasets.

In [21]:
# Get Llama 3.2 to answer

def stream_technical_answer_llama(question):
    stream = ollama.chat(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk['message']['content'] or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_technical_answer_llama(question)

**Code Explanation**

The given code snippet is written in Python and utilizes the `yield from` statement, which is part of the Generator protocol. Let's break it down:

python
yield from {book.get("author") for book in books if book.get("author")}


Here's what happens when this line is executed:

1. The expression `{...}` creates a dictionary comprehension.
2. `for book in books` iterates over the `books` collection (presumably an iterable, such as a list or tuple).
3. `if book.get("author")` filters out any items that don't have an "author" key present in each item. The `.get()` method returns `None` if the key is not found.
4. The expression `{...}` creates another dictionary comprehension, which generates a new dictionary containing only the "author"s from the books that passed the filter.

However, there's a crucial point to note: `yield from` is used in conjunction with a generator function (not shown in this snippet). When `yield from` is encountered inside a generator function, it yields the values generated by the expression after the `yield from` statement.

**Why**

In essence, this code achieves the following:

* Filters out books without an "author" key.
* For each book with an "author", extracts and yields the author's name.

The reason for using `yield from` is to create a generator that yields individual values (in this case, author names) instead of a list or other iterable. This has implications for performance and memory usage:

* The original code would create a new dictionary comprehension for each book in the `books` collection, which could lead to excessive memory allocation.
* By using `yield from`, we only generate one dictionary comprehension with all the authors' names at once, which can be more efficient.

**Example Usage**

Here's an example of how this code might be used:

python
# Define a generator function that yields author names
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Assume 'books' is a list of dictionaries with "author" keys
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2"},  # missing author key
    {"title": "Book 3", "author": "Jane Smith"}
]

# Create an iterator that yields authors' names
authors_iterator = get_authors(books)

# Iterate over the iterator to print the authors' names
for author in authors_iterator:
    print(author)  # Output: John Doe, Jane Smith


Note that this is a simplified example, and you may need to adapt it to your specific use case.